In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import random

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
import torchvision.models as models

import csv
import copy

from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
save_dir = "/content/drive/MyDrive/BreastCancerClassification/"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

log_file = os.path.join(save_dir, "training_log_new.csv")
best_model_path = os.path.join(save_dir, "best_model_new.pth")
last_model_path = os.path.join(save_dir, "last_model_new.pth")

In [ ]:
!unzip -q /content/drive/MyDrive/BreastCancerClassification/breastcancer.zip -d /content/cbis_images

In [ ]:
# A CBIS-DDSM gyárilag szét van választva, így nem kell train_test_split!
train_df = pd.read_csv("/content/cbis_images/csv/mass_case_description_train_set.csv")
test_df = pd.read_csv("/content/cbis_images/csv/mass_case_description_test_set.csv")

In [ ]:
# Ha a pathology oszlop tartalmazza a "MALIGNANT" szót, akkor 1 (Rákos).
# Minden más (BENIGN és BENIGN_WITHOUT_CALLBACK) 0 (Jóindulatú).
train_df["label"] = train_df["pathology"].apply(lambda x: 1 if "MALIGNANT" in str(x).upper() else 0)
test_df["label"] = test_df["pathology"].apply(lambda x: 1 if "MALIGNANT" in str(x).upper() else 0)

print(f"Tanító halmaz mérete: {len(train_df)} kép")
print("Tanító halmaz osztályeloszlása:\n", train_df["label"].value_counts())
print(f"\nTeszt halmaz mérete: {len(test_df)} kép")
print("Teszt halmaz osztályeloszlása:\n", test_df["label"].value_counts())

# Osztálysúlyok (Class weights) számítása Szigorúan a TRAIN halmazból
class_counts = train_df["label"].value_counts().sort_index()
total = len(train_df)
weights = total / (2 * class_counts)
class_weights = torch.tensor(weights.values, dtype=torch.float)

In [ ]:
# ImageNet átlag és szórás a normalizációhoz
normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

image_size = 512

train_transform = transforms.Compose([
    transforms.Resize((image_size + 32, image_size + 32)),
    transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), shear=5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5

    cm = confusion_matrix(y_true, y_pred)
    report_dict = classification_report(y_true, y_pred, labels=[0, 1], output_dict=True, zero_division=0)
    report_text = classification_report(y_true, y_pred, labels=[0, 1], target_names=['Jóindulatú (0)', 'Rosszindulatú (1)'], zero_division=0)

    return {
        "malignant_precision": report_dict['1']['precision'],
        "malignant_recall": report_dict['1']['recall'],
        "malignant_f1": report_dict['1']['f1-score'],
        "benign_precision": report_dict['0']['precision'],
        "benign_recall": report_dict['0']['recall'],
        "benign_f1": report_dict['0']['f1-score'],
        "auc": auc,
        "confusion_matrix": cm,
        "report_text": report_text
    }

In [ ]:
class BreastDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.valid_paths = [] # Itt tároljuk a már leellenőrzött útvonalakat

        print("Adatbázis indexelése és maszk-szűrés folyamatban... (ez tarthat pár percig)")
        for idx in tqdm(range(len(self.df))):
            row = self.df.iloc[idx]
            path_parts = str(row["cropped image file path"]).split('/')
            found_path = None

            for part in path_parts:
                folder_path = os.path.join(self.img_dir, part)
                if os.path.isdir(folder_path):
                    files = [f for f in os.listdir(folder_path) if f.endswith('.jpg')]
                    if len(files) > 1:
                        # Itt dől el egyszer és mindenkorra, melyik a jó kép
                        best_f = None
                        max_unique = -1
                        for f in files:
                            f_path = os.path.join(folder_path, f)
                            tmp = Image.open(f_path).convert('L')
                            u_count = len(np.unique(np.array(tmp)))
                            if u_count > max_unique:
                                max_unique = u_count
                                best_f = f_path
                        found_path = best_f
                    elif files:
                        found_path = os.path.join(folder_path, files[0])
                    break
            self.valid_paths.append(found_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Most már csak kiolvassuk az előre elmentett útvonalat
        img_path = self.valid_paths[idx]
        label = torch.tensor(self.df.iloc[idx]["label"], dtype=torch.float32)

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
train_dataset = BreastDataset(train_df, "/content/cbis_images/jpeg", train_transform)
val_dataset = BreastDataset(test_df, "/content/cbis_images/jpeg", val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
class AdvancedCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            # kisebbről indítjuk
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.4),

            # plusz 1 blokk kép nagyobb mérete miatt
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.5),
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.global_pool(x)
        x = self.fc(x)
        return x

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AdvancedCNN().to(device)

num_epochs = 50

pos_weight = torch.tensor([class_weights[1] / class_weights[0]]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=1e-6
)
print(pos_weight)

In [ ]:
def count_parameters(model):
    # Végigmegy a modell összes rétegén, és összeadja a tanítható (requires_grad=True) paraméterek számát
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

my_model = AdvancedCNN()
my_params = count_parameters(my_model)

print(f"Saját CNN paraméterszáma: {my_params:,}")

In [ ]:
start_epoch = 0
best_score = 0
epochs_no_improve = 0

if os.path.exists(last_model_path):
    print("-> Korábbi mentés megtalálva! Folytatás betöltése...")
    checkpoint = torch.load(last_model_path, weights_only=False)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    start_epoch = checkpoint['epoch'] + 1
    best_score = checkpoint['best_score']
    epochs_no_improve = checkpoint['epochs_no_improve']

    print(f"-> A tanítás a(z) {start_epoch + 1}. epochától folytatódik.")
else:
    print("-> Nincs korábbi mentés. Tanítás indítása a nulláról.")

In [ ]:
if start_epoch == 0:
    with open(log_file, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch", "train_loss", "test_loss",
            "malignant_precision", "malignant_recall", "malignant_f1",
            "benign_precision", "benign_recall", "benign_f1",
            "auc", "learning_rate"
        ])

In [ ]:
patience = 10

# --- 10. TANÍTÁSI CIKLUS ---
print("Tanítás futtatása...")
for epoch in range(start_epoch, num_epochs):

    # Train
    model.train()
    train_loss = 0
    train_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False)

    for images, labels in train_progress:
        images = images.to(device)
        labels = labels.to(device).float()

        outputs = model(images).squeeze(1)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_progress.set_postfix({'loss': f"{loss.item():.4f}"})

    train_loss /= len(train_loader)

    # Validáció
    model.eval()
    test_loss = 0
    all_preds, all_labels, all_probs = [], [], []
    test_progress = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", leave=False)

    with torch.no_grad():
        for images, labels in test_progress:
            images = images.to(device)
            labels = labels.to(device).float()

            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            probs = torch.sigmoid(outputs).cpu().numpy()
            preds = (probs > 0.5).astype(int)

            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            test_progress.set_postfix({'loss': f"{loss.item():.4f}"})

    test_loss /= len(val_loader)
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    metrics = compute_metrics(all_labels, all_preds, all_probs)

    print(f"\n{'-'*50}")
    print(f"Epoch {epoch+1}/{num_epochs} | LR: {current_lr}")
    print(f"Train loss: {train_loss:.4f} | Val loss: {test_loss:.4f} | AUC: {metrics['auc']:.4f}")
    print("\nClassification Report:\n", metrics["report_text"])
    print("Confusion matrix:\n", metrics["confusion_matrix"])
    print(f"{'-'*50}\n")

    with open(log_file, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch+1, train_loss, test_loss,
            metrics['malignant_precision'], metrics['malignant_recall'], metrics['malignant_f1'],
            metrics['benign_precision'], metrics['benign_recall'], metrics['benign_f1'],
            metrics['auc'], current_lr
        ])

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_score': best_score,
        'epochs_no_improve': epochs_no_improve
    }
    torch.save(checkpoint, last_model_path)

    score = metrics["auc"]
    if score > best_score:
        best_score = score
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        print("-> Új legjobb modell mentve (AUC javult)!")
    else:
        epochs_no_improve += 1
        print(f"-> Nincs javulás {epochs_no_improve} epocha óta.")

    if epochs_no_improve >= patience:
        print(f"Early stopping aktiválva az {epoch+1}. epochánál.")
        break

print("\nTanítás befejezve!")